# Эксперимент 32: ASR screening for children

**Статья:** Automatic Screening for Children with Speech Disorder using ASR: opportunities and challenges (Автоматический скрининг детей с речевыми нарушениями с использованием ASR: возможности и ограничения) 2024

**Ссылка:** [https://arxiv.org/abs/2410.11865](https://arxiv.org/abs/2410.11865)

**Краткое описание модели:** ASR-диагностические признаки из CTC logits (wav2vec2-base-960h): confidence/entropy/blank-ratio/token diversity + классификатор.

**Содержание статьи:** Реализован ASR-centered скрининг: извлекаются статистики неопределенности и структуры токен-потока из CTC модели для детекции речевых нарушений.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

I0000 00:00:1774798708.233622   39525 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [ ]:
MODEL_ID = "facebook/wav2vec2-base-960h"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device)
model.eval()
blank_id = model.config.pad_token_id

def extract_diag(path):
    y, sr = data_utils.load_audio(path, sr=16000, max_sec=config.MAX_DURATION_SEC)
    inp = processor(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(inp.input_values.to(device)).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
    blank_ratio = float(np.mean(pred == blank_id)) if blank_id is not None else 0.0
    nonblank = pred[pred != blank_id] if blank_id is not None else pred
    unique_ratio = float(len(np.unique(nonblank)) / max(1, len(nonblank)))
    speech_rate = float(len(nonblank) / max(1e-6, len(y) / 16000.0))
    margin = np.partition(probs, -2, axis=1)[:, -1] - np.partition(probs, -2, axis=1)[:, -2]
    return np.array([conf.mean(), conf.std(), conf.min(), entropy.mean(), entropy.std(), entropy.max(), blank_ratio, unique_ratio, speech_rate, margin.mean(), margin.std()], dtype=np.float32)

X_train = np.stack([extract_diag(p) for p in paths_train])
X_val   = np.stack([extract_diag(p) for p in paths_val])
X_test  = np.stack([extract_diag(p) for p in paths_test])

X_train = np.hstack([X_train, letters_train])
X_val   = np.hstack([X_val, letters_val])
X_test  = np.hstack([X_test, letters_test])


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 3. Обучение, оценка и сохранение результата

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=4000, solver="liblinear"))])
param_grid = {"clf__C": [0.1, 0.3, 1.0, 3.0, 10.0]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_32_asr_screening_children", 
    experiment_name="ASR screening (CTC diagnostics)", 
    model="CTC diagnostics + LogReg", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"wav2vec2_ctc_diagnostics/grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=None
)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
              precision    recall  f1-score   support

        good       0.76      0.68      0.72       282
         bad       0.45      0.56      0.50       135

    accuracy                           0.64       417
   macro avg       0.61      0.62      0.61       417
weighted avg       0.66      0.64      0.65       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.640288,0.609551,0.5,0.693538,0.454545,0.555556


Результат сохранён в result.csv текущего эксперимента
